# 02 Features, labels and the monthly panel

Work packages 2.4, 2.5 and 3.1 of the WBS. Starting from the clean daily prices of notebook 01, this notebook:

1. computes the seven indicators for every ETF and trading day,
2. samples them at each month-end and attaches the two labels (next month's return, and whether the ETF finishes in next month's top 3),
3. validates the panel, including a look-ahead test,
4. builds the walk-forward calendar (which rows train, tune and test each month),
5. checks every saved table against the data dictionary in `src/dictionary.py`,
6. explores the data using **pre-2020 rows only**, so nothing about the test period influences design choices.

Outputs go to `data/processed/` and `data/reports/` on Drive.

In [ ]:
# Setup: run this cell first in every Colab session.
# It mounts Google Drive (where the data lives), gets the latest code from
# GitHub (where the code lives) and installs the pinned libraries.
from google.colab import drive
drive.mount("/content/drive")

import os
import sys

REPO_URL = "https://github.com/guyen5/Capstone.git"
REPO_DIR = "/content/Capstone"

# First run in a session: clone the repo. Later runs: pull the newest code.
if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull --quiet
else:
    !git clone --quiet {REPO_URL} {REPO_DIR}

%pip install --quiet -r {REPO_DIR}/requirements.txt

# Make "from src import ..." work.
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

In [ ]:
# Libraries, project modules and folders.
import importlib
import json
from datetime import datetime, timezone

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src import config, data_pull, quality, features, splits, dictionary

# Reload so edits pulled from GitHub take effect without a runtime restart.
for module in (config, data_pull, quality, features, splits, dictionary):
    importlib.reload(module)

DIRS = config.data_dirs()
pd.set_option("display.width", 160)
print("Features:", config.FEATURES)
print("Test holding months:", config.TEST_FIRST_HOLDING, "to", config.TEST_LAST_HOLDING)

## Step 1. Load the clean daily prices

`prices_clean_v1.parquet` was built by notebook 01 from the frozen download. The freeze metadata is read so the panel records exactly which download it came from.

In [ ]:
clean = quality.load_clean(DIRS["processed"])
freeze_meta = json.loads(data_pull.list_freezes(DIRS["raw"])[-1].read_text())

print(f"{len(clean):,} rows, {clean['ticker'].nunique()} tickers, "
      f"{clean['date'].min():%Y-%m-%d} to {clean['date'].max():%Y-%m-%d}")
print("Built from freeze:", freeze_meta["file"], "md5", freeze_meta["md5"])

# The clean table must still match its dictionary entry.
quality.summarize(dictionary.check_table(clean, "prices_clean_v1").rename(columns={"column": "ticker"}))

## Step 2. Daily features

Seven indicators per ETF per trading day (formulas in the data dictionary). Every value at day t uses only prices and volumes up to day t.

In [ ]:
daily = features.daily_features(clean)
print(f"{len(daily):,} rows")
daily[daily["ticker"] == "SPY"].dropna().tail()

## Step 3. Monthly panel with labels

One row per universe, ETF and month-end from January 2010. `next_return` is label A (Model A), `top3_next` is label B (Model B). The newest month (September 2026) has features but no label yet.

In [ ]:
panel = features.build_panel(clean, daily)

print(panel.groupby(["universe", "sample"]).size().unstack(fill_value=0))
print()
print("Month-ends:", panel["date"].nunique(),
      f"({panel['date'].min():%Y-%m} to {panel['date'].max():%Y-%m})")
print("Volume bias filled with 0 (zero-volume month-ends):",
      int(panel["volume_bias_missing"].sum()), "rows",
      panel.loc[panel["volume_bias_missing"], ["date", "ticker"]].drop_duplicates()
      .assign(date=lambda d: d["date"].dt.strftime("%Y-%m-%d")).values.tolist())
panel.head()

## Step 4. Validate the panel

Structural checks, an independent recomputation of label A, and the look-ahead test: features at random month-ends are recomputed from prices cut off at that date and must match exactly.

In [ ]:
panel_checks = features.validate_panel(panel, clean)
dict_checks = dictionary.check_table(panel, "monthly_panel_v1")

print(panel_checks.to_string(index=False))
print()
print(dict_checks.to_string(index=False))

all_checks = pd.concat([panel_checks.assign(column=""), dict_checks], ignore_index=True)
if (all_checks["status"] == "FAIL").any():
    raise RuntimeError("Panel check failed. Do not save or model on this panel.")

## Step 5. Walk-forward calendar

For each of the 81 test months: the decision date, the training window (rows whose labels are known before the decision) and the 12-month validation slice. The growing window is the primary design; 36- and 60-month rolling windows are robustness checks.

In [ ]:
calendars = {w: splits.walk_forward_calendar(panel, w)
             for w in ["growing", *config.ROLLING_WINDOWS]}
calendar = pd.concat(calendars.values(), ignore_index=True)

for w, cal in calendars.items():
    print(f"window {w}:")
    print(splits.check_calendar(cal, panel).to_string(index=False))
    print()
print(dictionary.check_table(calendar, "walk_forward_calendar")["status"].value_counts().to_dict())
calendars["growing"].iloc[[0, 1, -1]]

## Step 6. Save

Parquet for code, CSV for reading. The JSON file records where the panel came from and the settings used to build it.

In [ ]:
out = DIRS["processed"]
panel.to_parquet(out / "monthly_panel_v1.parquet", index=False)
panel.to_csv(out / "monthly_panel_v1.csv", index=False)
calendar.to_parquet(out / "walk_forward_calendar_v1.parquet", index=False)
calendar.to_csv(out / "walk_forward_calendar_v1.csv", index=False)
dictionary.to_frame().to_csv(out / "data_dictionary_v1.csv", index=False)
all_checks.to_csv(DIRS["reports"] / "panel_checks_v1.csv", index=False)

meta = {
    "file": "monthly_panel_v1.parquet",
    "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "source_freeze": freeze_meta["file"],
    "source_freeze_md5": freeze_meta["md5"],
    "rows": {u: int(n) for u, n in panel["universe"].value_counts().items()},
    "month_ends": int(panel["date"].nunique()),
    "first_date": f"{panel['date'].min():%Y-%m-%d}",
    "last_date": f"{panel['date'].max():%Y-%m-%d}",
    "features": config.FEATURES,
    "top_k": config.TOP_K,
    "test_holding_months": [config.TEST_FIRST_HOLDING, config.TEST_LAST_HOLDING],
    "volume_bias_filled_rows": int(panel["volume_bias_missing"].sum()),
}
(out / "monthly_panel_v1.json").write_text(json.dumps(meta, indent=2))
print(json.dumps(meta, indent=2))

## Step 7. Exploration (pre-2020 rows only)

Everything below uses `sample == "pre_test"` rows of ETF_A, the primary universe. The test period (2020 onward) stays unseen until the primary specification is locked (WBS 4.3). These tables describe the data; none of them is used to choose features.

In [ ]:
pre = panel[(panel["sample"] == "pre_test") & (panel["universe"] == "ETF_A")].copy()
print(f"{pre['date'].nunique()} months, {len(pre)} rows "
      f"({pre['holding_month'].min()} to {pre['holding_month'].max()})")

# 7a. Feature summary.
summary = pre[config.FEATURES].describe(percentiles=[0.05, 0.5, 0.95]).T
summary.to_csv(DIRS["reports"] / "eda_feature_summary_pre2020.csv")
summary.round(4)

In [ ]:
# 7b. Who finishes in the top 3? Chance level is 3/8 = 37.5% for every ETF.
top3_rate = (pre.groupby("ticker")["top3_next"].mean().sort_values(ascending=False)
             .rename("share of months in top 3"))

# 7c. Persistence: if an ETF was in the top 3 this month, how often is it
# again next month? Well above 37.5% would mean rankings are sticky, which
# favours low-turnover strategies.
pre = pre.sort_values(["ticker", "date"])
pre["top3_now"] = pre.groupby("ticker")["top3_next"].shift(1)
stay = pre.dropna(subset=["top3_now"]).groupby("top3_now")["top3_next"].mean()

print(top3_rate.round(3).to_string())
print()
print(f"P(top 3 next month | top 3 this month)     = {stay.get(1.0, np.nan):.3f}")
print(f"P(top 3 next month | not top 3 this month) = {stay.get(0.0, np.nan):.3f}")
top3_rate.to_csv(DIRS["reports"] / "eda_top3_rate_pre2020.csv")

In [ ]:
# 7d. Information coefficient: Spearman correlation between each feature
# and next month's return across the 8 ETFs, computed every month and then
# averaged. t-stat = mean / (std / sqrt(months)). With 8 ETFs per month,
# single-month values are very noisy; the average is what matters.
def monthly_ic(df, feature):
    return df.groupby("date").apply(
        lambda g: g[feature].corr(g["next_return"], method="spearman"),
        include_groups=False)

ic = pd.DataFrame({f: monthly_ic(pre, f) for f in config.FEATURES})
ic_table = pd.DataFrame({
    "mean IC": ic.mean(),
    "std": ic.std(),
    "t-stat": ic.mean() / (ic.std() / np.sqrt(len(ic))),
    "share of months > 0": (ic > 0).mean(),
}).round(3)
ic_table.to_csv(DIRS["reports"] / "eda_feature_ic_pre2020.csv")
ic_table

In [ ]:
# 7e. How similar are the features to each other? Highly correlated pairs
# carry overlapping information (matters for the linear baselines).
corr = pre[config.FEATURES].corr(method="spearman")

fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr)), corr.columns, rotation=45, ha="right")
ax.set_yticks(range(len(corr)), corr.columns)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
fig.colorbar(im, ax=ax, shrink=0.8)
ax.set_title("Spearman correlation between features (ETF_A, pre-2020)")
plt.tight_layout()
plt.show()

In [ ]:
# 7f. How much does ranking matter? The gap between the average top-3 and
# the average bottom-5 next-month return is the most a perfect ranker could
# earn each month before fees.
gap = (pre.groupby(["date", "top3_next"])["next_return"].mean().unstack())
gap = gap[1] - gap[0]

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.bar(gap.index, gap * 100, width=20, color="#3b7d74")
ax.axhline(gap.mean() * 100, color="black", linewidth=1, linestyle="--",
           label=f"average {gap.mean():.2%} per month")
ax.set_ylabel("top 3 minus rest (%)")
ax.set_title("Monthly return gap between the true top 3 and the other 5 ETFs (pre-2020)")
ax.legend()
plt.tight_layout()
plt.show()

## Step 8. Summary for the process log

Copy this into the log. Then save the notebook to GitHub: File, Save a copy in GitHub, path `02_features_and_panel.ipynb`.

In [ ]:
status = "PASS" if (all_checks["status"] == "PASS").all() else "CHECK"
print(f"Panel:           monthly_panel_v1.parquet ({status})")
print(f"Source freeze:   {freeze_meta['file']} (md5 {freeze_meta['md5']})")
print(f"Month-ends:      {panel['date'].nunique()} "
      f"({panel['date'].min():%Y-%m} to {panel['date'].max():%Y-%m})")
for u, n in panel["universe"].value_counts().sort_index().items():
    print(f"Rows {u}:      {n:,}")
print(f"Test months:     {len(calendars['growing'])} "
      f"({config.TEST_FIRST_HOLDING} to {config.TEST_LAST_HOLDING})")
print(f"Checks:          {int((all_checks['status'] == 'PASS').sum())} passed, "
      f"{int((all_checks['status'] == 'FAIL').sum())} failed")
print(f"Volume bias:     {int(panel['volume_bias_missing'].sum())} rows filled with 0")